In [ ]:
import numpy as np
import pyshtools as pysh
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import cartopy.crs as ccrs
from tqdm import tqdm

# ==========================================
# 1. 初始化参数与网格 (Gauss-Legendre Grid)
# ==========================================
lmax = 255  # 注意：在纯 Python 环境中 lmax=1020 依然计算量庞大，如需快速测试可先降为 255
# 对于高斯-勒让德网格，pyshtools 默认的经纬度尺寸
nlats = lmax + 1
nlons = 2 * lmax + 1
print("网格大小:", nlats, nlons, nlats * nlons)

# 获取高斯-勒让德网格的经纬度坐标 (返回度数，需转为弧度)
lats_deg, _ = pysh.expand.GLQGridCoord(lmax)
lons_deg = np.linspace(0, 360, nlons, endpoint=False)
lats = np.deg2rad(lats_deg)
lons = np.deg2rad(lons_deg)

# ==========================================
# 2. 预计算拉普拉斯算子、滤波器及能量谱斜率
# ==========================================
# PySHTools 的系数数组形状为 (2, lmax+1, lmax+1)，分别对应 cos 和 sin 项
l_arr = np.arange(lmax + 1)
l_2d = np.zeros((lmax + 1, lmax + 1))
for l in range(lmax + 1):
    l_2d[l, :] = l
degree = np.stack([l_2d, l_2d])

# ∇² = -l(l+1)
lap = -degree * (degree + 1.0)
invlap = np.zeros_like(lap, dtype=float)
mask = degree > 0
invlap[mask] = 1.0 / lap[mask]

# 高斯滤波器与初始能量谱斜率
spec_filter = np.exp(-36.0 * (degree / lmax)**8.0)
slope = np.ones_like(degree, dtype=float)
slope[mask] = degree[mask]**(-1.0 / 3.0)

# ==========================================
# 3. 场初始化
# ==========================================
# 随机初始化物理网格数据，并变换到球谐空间
grid = np.random.randn(nlats, nlons)
# SHExpandGLQ：将物理网格转换为球谐系数
zeta_coeffs = pysh.expand.SHExpandGLQ(grid) * slope  
zeta_coeffs *= spec_filter  

dt = 0.5        # 时间步长
nu = 1e-9       # 粘性系数

# ==========================================
# 4. 动力学与物理量换算函数
# ==========================================
def streamfunction_from_zeta(zeta_coeffs_in):
    """从 ζ 系数计算 ψ 系数: ψ = ζ / -l(l+1)"""
    return zeta_coeffs_in * invlap

def velocity_from_streamfunction(psi_coeffs_in):
    """从 ψ 系数计算速度场 u"""
    # MakeGridGradient 返回在 GLQ 网格上的梯度 (d/dtheta, 1/sin(theta) d/dphi)
    # 对应于你的 sht.synth_grad 的输出顺序
    dtheta_psi, dphi_psi = pysh.expand.MakeGridGradient(psi_coeffs_in, grid='GLQ')
    u_phi, u_theta = dtheta_psi, dphi_psi
    return u_phi, u_theta

def velocity_from_zeta(zeta_coeffs_in):
    """从 ζ 系数计算速度场 u"""
    psi_coeffs_in = streamfunction_from_zeta(zeta_coeffs_in)
    return velocity_from_streamfunction(psi_coeffs_in)

def nonl(zeta_c):
    """计算非线性项以及耗散项"""
    # 1. 解 ∇²ψ = ζ
    psi_c = streamfunction_from_zeta(zeta_c)
    
    # 2. 计算 ψ 的梯度 -> 速度场
    u_phi, u_theta = velocity_from_streamfunction(psi_c)
    
    # 3. 计算 ζ 的梯度
    dtheta_zeta, dphi_zeta = pysh.expand.MakeGridGradient(zeta_c, grid='GLQ')
    dtheta_zeta *= -1  # 对应原代码中的 -∂/∂θ
    
    # 4. 计算非线性对流项 N = u ⋅ ∇ζ (Jacobian)
    adv = u_theta * dtheta_zeta + u_phi * dphi_zeta
    
    # 5. 回到频域 (SHT)
    adv_coeffs = pysh.expand.SHExpandGLQ(adv)
    adv_coeffs *= spec_filter
    
    return -adv_coeffs + nu * (zeta_c * lap)

# ==========================================
# 5. 时间推进 (RK4)
# ==========================================
# MakeGridGLQ: 从球谐系数重构回物理网格
sol = [pysh.expand.MakeGridGLQ(zeta_coeffs)]  
psi_grid_init = pysh.expand.MakeGridGLQ(streamfunction_from_zeta(zeta_coeffs))

# 计算初始能量 (球面积分近似)
energy = [np.sum(np.abs(sol[0] * psi_grid_init))]
e0 = energy[0]
num_steps = 2000

print("开始时间推进...")
with tqdm(total=num_steps, desc="时间推进") as pbar:
    for t in range(num_steps):
        # 4阶 Runge-Kutta
        k1 = dt * nonl(zeta_coeffs)
        k2 = dt * nonl(zeta_coeffs + 0.5 * k1)
        k3 = dt * nonl(zeta_coeffs + 0.5 * k2)
        k4 = dt * nonl(zeta_coeffs + k3)
        zeta_coeffs += (k1 + 2*k2 + 2*k3 + k4) / 6.0

        # 每 10 步存储一次
        if (t+1) % 10 == 0:
            current_zeta_grid = pysh.expand.MakeGridGLQ(zeta_coeffs)
            sol.append(current_zeta_grid)
            
            current_psi_grid = pysh.expand.MakeGridGLQ(streamfunction_from_zeta(zeta_coeffs))
            energy.append(np.sum(np.abs(current_zeta_grid * current_psi_grid)))
            
            pbar.set_postfix({"能量": energy[-1] / e0})
            pbar.update(10)

# ==========================================
# 6. 数据可视化
# ==========================================
# 绘制能量衰减图
plt.figure(figsize=(8, 4))
plt.plot(energy, label='能量')
plt.title("Energy Evolution")
plt.xlabel("Saved Steps")
plt.ylabel("Total Energy")
plt.legend()
plt.show()

# 绘制 Cartopy 球面湍流演化动画
fig = plt.figure(figsize=(8, 6))
ax = plt.axes(projection=ccrs.Orthographic(0, 0))
ax.set_global()
ax.set_facecolor('black')

# 生成绘图网格
xs, ys = np.meshgrid(np.rad2deg(lons), np.flip(np.rad2deg(lats)), indexing='ij')

# 初始速度场
u, v = velocity_from_zeta(pysh.expand.SHExpandGLQ(sol[0]))
umag = np.sqrt(u**2 + v**2)
zs = umag.T

mesh = ax.pcolormesh(xs, ys, zs, transform=ccrs.PlateCarree(), shading='auto', cmap='RdBu_r')

def update_mesh(t):
    u_t, v_t = velocity_from_zeta(pysh.expand.SHExpandGLQ(sol[t]))
    umag_t = np.sqrt(u_t**2 + v_t**2)
    mesh.set_array(umag_t.T.flatten())
    ax.set_title(f'Velocity Evolution (step={t*10})')
    return mesh,

ani = FuncAnimation(fig, update_mesh, frames=len(sol), interval=100, blit=False)
plt.show()

# ==========================================
# 注意：如果你在 Jupyter Notebook 中运行这段代码，
# 请取消注释以下两行以直接内嵌 HTML 动画播放器：
# ==========================================
# from IPython.display import HTML
# HTML(ani.to_jshtml())